# 05 — EBM Explainability & SHAP-vs-EBM Comparison

The Explainable Boosting Machine (EBM) is a glass-box generalized additive model: its own term importances and shape functions *are* the explanation, computed exactly (not approximated) by the model itself — unlike SHAP, which is a post-hoc approximation applied to an otherwise opaque model (XGBoost, CatBoost). This notebook produces EBM's native explanation and measures how well it agrees, in relative feature ranking, with XGBoost's SHAP importances (RQ2).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import joblib

from dac.config import CONFIG
from dac.explainability.ebm_explain import compute_shap_ebm_agreement, explain_ebm_model

ebm_pipeline = joblib.load(CONFIG["paths"]["models_dir"] / "ebm_tuned.joblib")
ebm_importance = explain_ebm_model(
    ebm_pipeline, "ebm_tuned", CONFIG["paths"]["figures_dir"] / "uci_credit" / "ebm",
)
ebm_importance.head(20)

## Agreement with XGBoost's SHAP importances

In [ ]:
import pandas as pd

shap_csv = CONFIG["paths"]["figures_dir"] / "uci_credit" / "shap" / "xgboost_tuned_shap_importance.csv"
shap_importance = pd.read_csv(shap_csv, index_col=0)["mean_abs_shap"]

agreement = compute_shap_ebm_agreement(shap_importance, ebm_importance["ebm_importance"])
agreement

A high, statistically significant Spearman correlation here means the post-hoc SHAP approximation (applied to XGBoost) and EBM's exact, native explanation are picking out the same underlying drivers of default risk — evidence that SHAP's approximation is trustworthy for this dataset, not just self-consistent. See `reports/figures/uci_credit/ebm/*_shape_*.png` for the individual shape-function plots referenced in the Interim Report.